In [ ]:
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"
import xgboost as xgb

In [ ]:
# ============================================================
# SHAP-set (XGBoost version): single-file implementation
# + CyTOF/EpiTOF-like simulation
# ============================================================
# pip install numpy pandas anndata scanpy shap xgboost scikit-learn scipy

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from tqdm import tqdm
from typing import Dict, Set
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.calibration import calibration_curve
from scipy.stats import chi2, norm
from xgboost import XGBClassifier

# ---------------------------
# Utility helpers
# ---------------------------
def get_layer_matrix(adata, layer: str):
    """Return matrix from AnnData layer or X."""
    return np.asarray(adata.layers[layer] if (layer and layer in adata.layers) else adata.X, dtype=np.float32)

def intersect_sets(var_names, marker_sets: Dict[str, Set[str]]):
    """Intersect sets with current feature space; return names, mask, sizes."""
    var_names = np.asarray(var_names)
    names, masks, sizes = [], [], []
    for nm, genes in marker_sets.items():
        m = np.isin(var_names, list(genes))
        if m.any():
            names.append(nm)
            masks.append(m)
            sizes.append(int(m.sum()))
    if not names:
        raise ValueError("No marker/gene sets overlap the features (var_names).")
    Gmask = np.vstack(masks).astype(bool)
    return names, Gmask, np.asarray(sizes, int)

def bh_fdr(p):
    """Benjamini–Hochberg FDR."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order]
    n = len(p)
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    return out

def oof_classifier_metrics_xgb(X, y_int, n_splits=5, seed=0, model_kwargs=None):
    """XGBoost OOF metrics (macro/micro F1, macro AUROC, log loss, Brier, ECE)."""
    from sklearn.metrics import log_loss, confusion_matrix
    model_kwargs = model_kwargs or {}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    base = XGBClassifier(
        objective="multi:softprob",
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=0,
        **model_kwargs,
    )
    C = len(np.unique(y_int))
    probs = np.zeros((len(y_int), C))
    preds = np.empty(len(y_int), dtype=int)
    for tr, te in skf.split(X, y_int):
        clf = XGBClassifier(**base.get_params())
        clf.set_params(random_state=int(np.random.default_rng(seed).integers(1, 2**31 - 1)))
        clf.fit(X[tr], y_int[tr])
        P = clf.predict_proba(X[te])  # shape: (n_te, C)
        probs[te] = P
        preds[te] = np.argmax(P, axis=1)
    macro_f1 = f1_score(y_int, preds, average="macro")
    micro_f1 = f1_score(y_int, preds, average="micro")
    Y = pd.get_dummies(y_int)
    try:
        macro_auroc = roc_auc_score(Y, probs, average="macro", multi_class="ovr")
    except ValueError:
        macro_auroc = np.nan
    ll = log_loss(y_int, probs)
    # Brier score (one-vs-rest averaged)
    brier = np.mean([( (y_int == i).astype(int) - probs[:, i]) ** 2 for i in range(C)])
    # ECE on maxprob
    conf = probs.max(1); acc = (preds == y_int).astype(int)
    fracs, means = calibration_curve(acc, conf, n_bins=10, strategy="quantile")
    ece = float(np.mean(np.abs(fracs - means)))
    cm = pd.DataFrame(
        confusion_matrix(y_int, preds, labels=np.arange(C)),
        index=[f"true_{i}" for i in range(C)],
        columns=[f"pred_{i}" for i in range(C)],
    )
    return {
        "classes_int": np.arange(C),
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_auroc": macro_auroc,
        "log_loss": ll,
        "brier": brier,
        "ECE": ece,
        "confusion_matrix": cm,
        "oof_probs": probs,
        "oof_preds": preds,
    }

# Robust SHAP tensor coercion
def to_shap_tensor(sv):
    """
    Coerce SHAP outputs to array of shape [cells, classes, features].
    Handles (list of C arrays), (cells, features) for binary, and 3D variants.
    """
    import numpy as _np
    if isinstance(sv, list):
        return _np.stack(sv, axis=1)  # (cells, C, G)
    arr = _np.asarray(sv)
    if arr.ndim == 2:  # (cells, G) binary
        return _np.stack([-arr, arr], axis=1)  # (cells, 2, G)
    if arr.ndim == 3:
        # (C, cells, G) -> (cells, C, G)
        if arr.shape[0] <= 10 and arr.shape[1] >= 10:
            return _np.transpose(arr, (1, 0, 2))
        # (cells, G, C) -> (cells, C, G)
        if arr.shape[2] <= 10 and arr.shape[0] >= 10:
            return _np.transpose(arr, (0, 2, 1))
        return arr
    raise ValueError(f"Unexpected SHAP output shape: {arr.shape}")

# ---------------------------
# Rank-based & competitive-null per-cell set scores
# ---------------------------
def aucell_scores(adata, marker_sets: Dict[str, Set[str]], layer="arcsinh", top_prop=0.05):
    X = get_layer_matrix(adata, layer)
    G = X.shape[1]
    top_n = max(1, int(round(G * top_prop)))
    genes = np.asarray(adata.var_names)
    set_names, Gmask, sizes = intersect_sets(genes, marker_sets)
    order = np.argsort(-X, axis=1)
    ranks = np.empty_like(order, dtype=np.int32)
    rows = np.arange(X.shape[0])[:, None]
    pos = np.arange(G)[None, :]
    ranks[rows, order] = pos + 1
    in_top = (ranks <= top_n)
    M = np.zeros((X.shape[0], len(set_names)), dtype=np.float32)
    for s in range(len(set_names)):
        M[:, s] = (in_top[:, Gmask[s]].sum(1) / float(top_n)).astype(np.float32)
    return pd.DataFrame(M, index=adata.obs_names, columns=set_names)

def ucell_scores(adata, marker_sets: Dict[str, Set[str]], layer="arcsinh"):
    X = get_layer_matrix(adata, layer)
    G = X.shape[1]
    genes = np.asarray(adata.var_names)
    set_names, Gmask, sizes = intersect_sets(genes, marker_sets)
    order = np.argsort(-X, axis=1)
    ranks = np.empty_like(order, dtype=np.int32)
    rows = np.arange(X.shape[0])[:, None]
    pos = np.arange(G)[None, :]
    ranks[rows, order] = pos + 1
    M = np.zeros((X.shape[0], len(set_names)), dtype=np.float32)
    for s, m in enumerate(sizes):
        r = ranks[:, Gmask[s]]
        U = r.sum(1) - (m * (m + 1)) / 2.0
        M[:, s] = (U / (m * (G - m))).astype(np.float32) if m < G else 1.0
    return pd.DataFrame(M, index=adata.obs_names, columns=set_names)

def sipsic_like_scores(adata, marker_sets: Dict[str, Set[str]], layer="arcsinh", n_perm=500, seed=0):
    rng = np.random.default_rng(seed)
    X = get_layer_matrix(adata, layer)
    genes = np.asarray(adata.var_names)
    set_names, Gmask, sizes = intersect_sets(genes, marker_sets)
    G = X.shape[1]
    Xc = X - X.mean(axis=1, keepdims=True)
    obs = Xc @ Gmask.T
    uniq = np.unique(sizes)
    mu = np.zeros((X.shape[0], len(uniq)))
    sd = np.zeros_like(mu)
    for t, m in enumerate(uniq):
        idxs = np.stack([rng.choice(G, size=m, replace=False) for _ in range(n_perm)], axis=0)
        null = Xc[:, idxs].sum(axis=2)
        mu[:, t] = null.mean(1)
        sd[:, t] = null.std(1) + 1e-6
    size_to_pos = {m: i for i, m in enumerate(uniq)}
    Z = np.zeros_like(obs)
    P = np.ones_like(obs, dtype=float)
    for s, m in enumerate(sizes):
        t = size_to_pos[m]
        Z[:, s] = (obs[:, s] - mu[:, t]) / sd[:, t]
        P[:, s] = norm.sf(Z[:, s])
    return (
        pd.DataFrame(Z, index=adata.obs_names, columns=set_names),
        pd.DataFrame(P, index=adata.obs_names, columns=set_names),
    )

# ---------------------------
# SHAP-set (OOF CV, mean-SHAP, Rank-SHAP, signed fraction, perm Z/p/FDR) with XGBoost
# ---------------------------
def shap_cluster_gene_set_scores_xgb(
    adata,
    marker_sets: Dict[str, Set[str]],
    layer: str = "arcsinh",
    cluster_key: str = "cluster",
    do_cluster: bool = False,
    leiden_res: float = 1.0,
    neighbors_k: int = 15,
    n_repeats: int = 2,
    n_splits: int = 5,
    n_perm: int = 200,
    random_state: int = 0,
    model_kwargs: dict | None = None,
    signed_weights: Dict[str, Dict[str, float]] | None = None,
    return_metrics: bool = True,
    class_agg: str = "prob",   # <-- NEW: "prob" | "mean" | "max" | "pred"
):
    import shap
    from xgboost import XGBClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder

    rng_master = np.random.default_rng(random_state)
    print("Start Scoring")
    # pseudo-labels (optional)
    if (cluster_key not in adata.obs) and do_cluster:
        rep = "X_cytovi" if "X_cytovi" in adata.obsm else None
        print(f"Calculating kNN with {neighbors_k} neighbors")
        sc.pp.neighbors(adata, use_rep=rep, n_neighbors=neighbors_k)
        print(f"Clustering with resolution {leiden_res}")
        sc.tl.leiden(adata, key_added=cluster_key, resolution=leiden_res)

    # encode labels for XGBoost
    y_str = adata.obs[cluster_key].astype(str).to_numpy()
    le = LabelEncoder().fit(y_str)
    y_int = le.transform(y_str)
    class_names = le.classes_

    # features (optionally restrict to union of set members)
    X_all = get_layer_matrix(adata, layer)
    genes_all = np.asarray(adata.var_names)
    union = np.zeros(len(genes_all), bool)
    for S in marker_sets.values():
        union |= np.isin(genes_all, list(S))
    if union.sum() >= 5:
        X = X_all[:, union]; genes = genes_all[union]
    else:
        X = X_all; genes = genes_all

    set_names, Gmask, sizes = intersect_sets(genes, marker_sets)
    S = len(set_names); G = X.shape[1]

    # optional signed weights
    W = None
    if signed_weights is not None:
        W = np.zeros((S, G), dtype=np.float32)
        for si, name in enumerate(set_names):
            if name in signed_weights:
                for feat, w in signed_weights[name].items():
                    idx = np.where(genes == feat)[0]
                    if idx.size: W[si, idx[0]] = float(w)

    # CV model
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    base = XGBClassifier(
        objective="multi:softprob",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=0,
        **(model_kwargs or {}),
    )

    # containers across repeats
    mean_runs = []; p_runs = []; rank_runs = []; frac_runs = []; z_runs = []
    prob_runs = []; pred_runs = []  # OOF probabilities & argmax preds for aggregation

    for rep in range(n_repeats):
        rng = np.random.default_rng(rng_master.integers(1, 2**31 - 1))
        mean_oof = p_oof = rank_oof = frac_oof = z_oof = None
        oof_probs = np.zeros((X.shape[0], len(class_names)))
        oof_pred  = np.zeros(X.shape[0], dtype=int)

        for tr, te in tqdm(skf.split(X, y_int)):
            clf = XGBClassifier(**base.get_params())
            clf.set_params(random_state=int(rng.integers(1, 2**31 - 1)))
            clf.fit(X[tr], y_int[tr])
            P = clf.predict_proba(X[te])  # (te, C)
            oof_probs[te] = P
            oof_pred[te] = np.argmax(P, axis=1)

            # SHAP values
            expl = shap.TreeExplainer(clf, feature_names=genes)
            sv = expl.shap_values(X[te])
            SH = to_shap_tensor(sv)  # (cells_te, C, G)
            C = SH.shape[1]
            if mean_oof is None:
                shape = (X.shape[0], C, S)
                mean_oof = np.zeros(shape, np.float32)
                p_oof    = np.ones(shape, float)
                rank_oof = np.zeros(shape, np.float32)
                frac_oof = np.zeros(shape, np.float32)
                z_oof    = np.zeros(shape, np.float32)

            # (A) aggregate set sum
            set_sum = np.einsum("ckg,sg->cks", SH, Gmask.astype(np.float32), optimize=True)
            if W is not None:
                use = np.where(W.sum(1) != 0)[0]
                if use.size:
                    set_sum_w = np.einsum("ckg,sg->cks", SH, W.astype(np.float32), optimize=True)
                    set_sum[:, :, use] = set_sum_w[:, :, use]
            set_mean = set_sum / sizes[None, None, :].clip(min=1)

            # (B) signed fraction
            denom = np.abs(SH).sum(2, keepdims=True) + 1e-12
            set_frac = (set_sum / denom).astype(np.float32)

            # (C) Rank-SHAP
            order = np.argsort(-SH, axis=2)
            ranks = np.empty_like(order, np.int32)
            rows = np.arange(SH.shape[0])[:, None, None]
            cols = np.arange(C)[None, :, None]
            pos = np.arange(G)[None, None, :]
            ranks[rows, cols, order] = pos + 1
            set_rank = np.zeros_like(set_mean, np.float32)
            for s, m in enumerate(sizes):
                if m == 0: continue
                if m == G:
                    set_rank[:, :, s] = 1.0
                    continue
                r = ranks[:, :, Gmask[s]]
                U = r.sum(2) - (m * (m + 1)) / 2.0
                set_rank[:, :, s] = (U / (m * (G - m))).astype(np.float32)

            # (D) permutation Z & p
            uniq = np.unique(sizes)
            idx_pool = {m: np.stack([rng.choice(G, size=m, replace=False) for _ in range(n_perm)], 0)
                        for m in uniq}
            set_p = np.ones_like(set_mean)
            set_z = np.zeros_like(set_mean, np.float32)
            for s, m in enumerate(sizes):
                obs = set_sum[:, :, s]
                idxs = idx_pool[m]
                null = SH[:, :, idxs].sum(3)
                mu = null.mean(2)
                sd = null.std(2) + 1e-6
                set_z[:, :, s] = ((obs - mu) / sd).astype(np.float32)
                ge = (null >= obs[:, :, None]).sum(2)
                set_p[:, :, s] = (ge + 1.0) / (n_perm + 1.0)

            # write OOF slice
            mean_oof[te, :, :] = set_mean
            frac_oof[te, :, :] = set_frac
            rank_oof[te, :, :] = set_rank
            z_oof[te, :, :]    = set_z
            p_oof[te, :, :]    = set_p

        mean_runs.append(mean_oof); p_runs.append(p_oof)
        rank_runs.append(rank_oof); frac_runs.append(frac_oof); z_runs.append(z_oof)
        prob_runs.append(oof_probs); pred_runs.append(oof_pred)

    # combine repeats
    mean_arr = np.stack(mean_runs, 0).mean(0)   # [cells, C, S]
    rank_arr = np.stack(rank_runs, 0).mean(0)
    frac_arr = np.stack(frac_runs, 0).mean(0)
    z_arr    = np.stack(z_runs, 0).mean(0)
    p_arr    = np.stack(p_runs, 0)              # [R, cells, C, S]

    # combine p over repeats
    stat = -2.0 * np.sum(np.log(np.clip(p_arr, 1e-300, 1.0)), axis=0)
    df = 2 * p_arr.shape[0]
    p_comb = 1.0 - chi2.cdf(stat, df)           # [cells, C, S]
    fdr_comb = bh_fdr(p_comb.ravel()).reshape(p_comb.shape)

    # ===== NEW: class-agnostic aggregation to [cells, S] =====
    probs_mean = np.stack(prob_runs, 0).mean(0)   # [cells, C]
    pred_mode  = np.round(np.stack(pred_runs, 0).mean(0)).astype(int)  # crude consensus

    def agg_class(T):  # T: [cells, C, S] -> [cells, S]
        if class_agg == "prob":
            W = probs_mean / (probs_mean.sum(1, keepdims=True) + 1e-12)
            return (T * W[:, :, None]).sum(1)
        elif class_agg == "mean":
            return T.mean(1)
        elif class_agg == "max":
            return T.max(1)
        elif class_agg == "pred":
            rows = np.arange(T.shape[0])
            return T[rows, pred_mode, :]
        else:
            raise ValueError("class_agg must be one of {'prob','mean','max','pred'}")

    mean_cell = agg_class(mean_arr)
    rank_cell = agg_class(rank_arr)
    frac_cell = agg_class(frac_arr)
    z_cell    = agg_class(z_arr)
    p_cell    = agg_class(p_comb)
    fdr_cell  = agg_class(fdr_comb)

    # store BOTH: per-class (as before) and class-aggregated (new)
    for j, cls_name in enumerate(class_names):
        adata.obsm[f"shapset_mean_{cls_name}"]      = pd.DataFrame(mean_arr[:, j, :], index=adata.obs_names, columns=set_names)
        adata.obsm[f"shapset_rankucell_{cls_name}"] = pd.DataFrame(rank_arr[:, j, :], index=adata.obs_names, columns=set_names)
        adata.obsm[f"shapset_frac_{cls_name}"]      = pd.DataFrame(frac_arr[:, j, :], index=adata.obs_names, columns=set_names)
        adata.obsm[f"shapset_z_{cls_name}"]         = pd.DataFrame(z_arr[:, j, :],   index=adata.obs_names, columns=set_names)
        adata.obsm[f"shapset_pval_{cls_name}"]      = pd.DataFrame(p_comb[:, j, :],  index=adata.obs_names, columns=set_names)
        adata.obsm[f"shapset_fdr_{cls_name}"]       = pd.DataFrame(fdr_comb[:, j, :],index=adata.obs_names, columns=set_names)

    # NEW: class-agnostic matrices (cells × sets)
    adata.obsm["shapset_mean"]      = pd.DataFrame(mean_cell, index=adata.obs_names, columns=set_names)
    adata.obsm["shapset_rankucell"] = pd.DataFrame(rank_cell, index=adata.obs_names, columns=set_names)
    adata.obsm["shapset_frac"]      = pd.DataFrame(frac_cell, index=adata.obs_names, columns=set_names)
    adata.obsm["shapset_z"]         = pd.DataFrame(z_cell,    index=adata.obs_names, columns=set_names)
    adata.obsm["shapset_pval"]      = pd.DataFrame(p_cell,    index=adata.obs_names, columns=set_names)
    adata.obsm["shapset_fdr"]       = pd.DataFrame(fdr_cell,  index=adata.obs_names, columns=set_names)

    out = {
        "classes": class_names,
        "set_names": set_names,
        "scores_mean": mean_arr, "scores_rank": rank_arr,
        "scores_frac": frac_arr, "scores_z": z_arr,
        "pvals_cell_classwise": p_comb, "fdr_cell_classwise": fdr_comb,
        # new aggregated (cells × sets)
        "scores_mean_cell": mean_cell, "scores_rank_cell": rank_cell,
        "scores_frac_cell": frac_cell, "scores_z_cell": z_cell,
        "pvals_cell": p_cell, "fdr_cell": fdr_cell,
        "class_agg": class_agg,
    }

    if return_metrics:
        out["clf_metrics"] = oof_classifier_metrics_xgb(
            X, y_int, n_splits=n_splits, seed=random_state, model_kwargs=model_kwargs
        )
    return out



######### With absolute variant
def shap_cluster_gene_set_scores_xgb(
    adata,
    marker_sets: dict,
    layer: str = "arcsinh",
    cluster_key: str = "cluster",
    do_cluster: bool = False,
    leiden_res: float = 1.0,
    neighbors_k: int = 15,
    n_repeats: int = 2,
    n_splits: int = 5,
    n_perm: int = 200,
    random_state: int = 0,
    model_kwargs: dict | None = None,
    signed_weights: dict | None = None,
    return_metrics: bool = True,
    class_agg: str = "prob",
):
    """
    Per-cell, per-gene-set SHAP scoring using XGBoost (multiclass).
    Produces per-class matrices AND class-agnostic (cells × sets) matrices.
    Also computes absolute-value SHAP set Z-scores to avoid cancellation.
    """
    import numpy as np, pandas as pd, shap
    from xgboost import XGBClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from scipy.stats import chi2
    from tqdm import tqdm

    rng_master = np.random.default_rng(random_state)

    # Optional clustering
    if (cluster_key not in adata.obs) and do_cluster:
        import scanpy as sc
        rep = "X_cytovi" if "X_cytovi" in adata.obsm else None
        print(f"Calculating NN {neighbors_k}")
        sc.pp.neighbors(adata, use_rep=rep, n_neighbors=neighbors_k)
        print(f"Clustering resolution {leiden_res}")
        sc.tl.leiden(adata, key_added=cluster_key, resolution=leiden_res)

    # Encode labels
    y_str = adata.obs[cluster_key].astype(str).to_numpy()
    le = LabelEncoder().fit(y_str)
    y_int = le.transform(y_str)
    class_names = le.classes_
    C_master = len(class_names)

    # Feature matrix
    X_all = get_layer_matrix(adata, layer)
    genes_all = np.asarray(adata.var_names)
    union = np.zeros(len(genes_all), bool)
    for S in marker_sets.values():
        union |= np.isin(genes_all, list(S))
    if union.sum() >= 5:
        X = X_all[:, union]; genes = genes_all[union]
    else:
        X = X_all; genes = genes_all

    # Master sets
    set_names_master, _, _ = intersect_sets(genes_all, marker_sets)
    if len(set_names_master) == 0:
        raise ValueError("None of the marker sets overlap features.")
    S_master = len(set_names_master)
    name_to_master = {nm: i for i, nm in enumerate(set_names_master)}

    # CV model
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    base = XGBClassifier(objective="multi:softprob", n_estimators=500,
                         learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         reg_lambda=1.0, tree_method="hist", random_state=0,
                         **(model_kwargs or {}))

    # Containers across repeats
    mean_runs, p_runs, rank_runs, frac_runs, z_runs = [], [], [], [], []
    z_abs_runs, p_abs_runs = [], []   # <<< NEW for absolute variant
    prob_runs, pred_runs = [], []

    N = X.shape[0]
    for rep in range(n_repeats):
        rng = np.random.default_rng(rng_master.integers(1, 2**31 - 1))
        mean_oof = np.zeros((N, C_master, S_master), np.float32)
        rank_oof = np.zeros((N, C_master, S_master), np.float32)
        frac_oof = np.zeros((N, C_master, S_master), np.float32)
        z_oof    = np.zeros((N, C_master, S_master), np.float32)
        p_oof    = np.ones( (N, C_master, S_master), float)
        # <<< NEW
        z_oof_abs= np.zeros((N, C_master, S_master), np.float32)
        p_oof_abs= np.ones( (N, C_master, S_master), float)

        oof_probs = np.zeros((N, C_master))
        oof_pred  = np.zeros(N, dtype=int)

        for tr, te in tqdm(skf.split(X, y_int)):
            clf = XGBClassifier(**base.get_params())
            clf.set_params(random_state=int(rng.integers(1, 2**31 - 1)))
            clf.fit(X[tr], y_int[tr])
            P = clf.predict_proba(X[te])
            classes_fold = getattr(clf, "classes_", np.arange(P.shape[1]))
            class_pos = np.full(P.shape[1], -1, dtype=int)
            for i, c in enumerate(classes_fold):
                c = int(c)
                if 0 <= c < C_master:
                    class_pos[i] = c
                    oof_probs[te, c] = P[:, i]
            oof_pred[te] = np.argmax(oof_probs[te], axis=1)

            expl = shap.TreeExplainer(clf, feature_names=genes.tolist())
            sv = expl.shap_values(X[te])
            SH = to_shap_tensor(sv)     # (cells_te, C_sh, G_sh)
            C_sh, G_sh = SH.shape[1], SH.shape[2]

            if C_sh != len(classes_fold):
                classes_fold = np.arange(C_sh)
            class_pos_sh = np.full(C_sh, -1, dtype=int)
            for i, c in enumerate(classes_fold):
                c = int(c)
                if 0 <= c < C_master:
                    class_pos_sh[i] = c

            feats_axis = (np.asarray(expl.feature_names)
                          if getattr(expl, "feature_names", None) is not None and
                             len(expl.feature_names) == G_sh
                          else np.asarray(genes[:G_sh]))
            fold_set_names, Gmask_fold, sizes_fold = intersect_sets(feats_axis, marker_sets)
            if len(fold_set_names) == 0:
                continue
            fold_to_master = np.array([name_to_master[nm] for nm in fold_set_names], int)

            # signed mean
            set_sum = np.einsum("ckg,sg->cks", SH, Gmask_fold.astype(np.float32))
            set_mean = set_sum / sizes_fold[None,None,:].clip(min=1)

            # abs mean (NEW)
            set_sum_abs = np.einsum("ckg,sg->cks", np.abs(SH), Gmask_fold.astype(np.float32))
            set_mean_abs = set_sum_abs / sizes_fold[None,None,:].clip(min=1)

            # fraction
            denom = np.abs(SH).sum(2, keepdims=True) + 1e-12
            set_frac = (set_sum / denom).astype(np.float32)

            # rank (same as before)
            order = np.argsort(-SH, axis=2)
            ranks = np.empty_like(order, np.int32)
            rows = np.arange(SH.shape[0])[:, None, None]
            cols = np.arange(C_sh)[None, :, None]
            pos  = np.arange(G_sh)[None, None, :]
            ranks[rows, cols, order] = pos + 1
            set_rank = np.zeros_like(set_mean, np.float32)
            for s_idx, m in enumerate(sizes_fold):
                if m == 0: continue
                r = ranks[:, :, Gmask_fold[s_idx]]
                U = r.sum(2) - (m * (m+1))/2.0
                set_rank[:,:,s_idx] = (U / (m*(G_sh-m))).astype(np.float32)

            # perm tests
            uniq = np.unique(sizes_fold)
            idx_pool = {m: np.stack([rng.choice(G_sh, size=m, replace=False)
                                     for _ in range(n_perm)],0) for m in uniq}
            set_p = np.ones_like(set_mean); set_z = np.zeros_like(set_mean)
            set_p_abs = np.ones_like(set_mean_abs); set_z_abs = np.zeros_like(set_mean_abs)
            for s_idx, m in enumerate(sizes_fold):
                obs  = set_sum[:, :, s_idx]
                obs_abs = set_sum_abs[:, :, s_idx]
                idxs = idx_pool[m]
                null = SH[:, :, idxs].sum(3)
                mu = null.mean(2); sd = null.std(2)+1e-6
                set_z[:,:,s_idx] = ((obs-mu)/sd).astype(np.float32)
                ge = (null>=obs[:,:,None]).sum(2)
                set_p[:,:,s_idx] = (ge+1)/(n_perm+1)
                # abs
                null_abs = np.abs(SH)[:, :, idxs].sum(3)
                mu_a = null_abs.mean(2); sd_a = null_abs.std(2)+1e-6
                set_z_abs[:,:,s_idx] = ((obs_abs-mu_a)/sd_a).astype(np.float32)
                ge_a = (null_abs>=obs_abs[:,:,None]).sum(2)
                set_p_abs[:,:,s_idx] = (ge_a+1)/(n_perm+1)

            for i_c, cpos in enumerate(class_pos_sh):
                if cpos < 0: continue
                idx = np.ix_(te, [cpos], fold_to_master)
                mean_oof[idx] = set_mean[:,i_c,:][:,None,:]
                rank_oof[idx] = set_rank[:,i_c,:][:,None,:]
                frac_oof[idx] = set_frac[:,i_c,:][:,None,:]
                z_oof[idx]    = set_z[:,i_c,:][:,None,:]
                p_oof[idx]    = set_p[:,i_c,:][:,None,:]
                # abs variant
                z_oof_abs[idx]= set_z_abs[:,i_c,:][:,None,:]
                p_oof_abs[idx]= set_p_abs[:,i_c,:][:,None,:]

        mean_runs.append(mean_oof); rank_runs.append(rank_oof)
        frac_runs.append(frac_oof); z_runs.append(z_oof); p_runs.append(p_oof)
        z_abs_runs.append(z_oof_abs); p_abs_runs.append(p_oof_abs)
        prob_runs.append(oof_probs); pred_runs.append(oof_pred)

    # combine repeats
    mean_arr = np.stack(mean_runs,0).mean(0)
    rank_arr = np.stack(rank_runs,0).mean(0)
    frac_arr = np.stack(frac_runs,0).mean(0)
    z_arr    = np.stack(z_runs,0).mean(0)
    p_arr    = np.stack(p_runs,0)
    z_abs_arr= np.stack(z_abs_runs,0).mean(0)
    p_abs_arr= np.stack(p_abs_runs,0)

    stat = -2*np.sum(np.log(np.clip(p_arr,1e-300,1.0)),0)
    p_comb = 1-chi2.cdf(stat,2*p_arr.shape[0])
    fdr_comb = bh_fdr(p_comb.ravel()).reshape(p_comb.shape)

    stat_abs = -2*np.sum(np.log(np.clip(p_abs_arr,1e-300,1.0)),0)
    p_comb_abs = 1-chi2.cdf(stat_abs,2*p_abs_arr.shape[0])
    fdr_comb_abs = bh_fdr(p_comb_abs.ravel()).reshape(p_comb_abs.shape)

    probs_mean = np.stack(prob_runs,0).mean(0)
    pred_mode = np.round(np.stack(pred_runs,0).mean(0)).astype(int)
    def agg_class(T):
        if class_agg=="prob":
            Wc = probs_mean/(probs_mean.sum(1,keepdims=True)+1e-12)
            return (T*Wc[:,:,None]).sum(1)
        elif class_agg=="mean": return T.mean(1)
        elif class_agg=="max": return T.max(1)
        elif class_agg=="pred":
            rows=np.arange(T.shape[0]); return T[rows,pred_mode,:]
    mean_cell = agg_class(mean_arr)
    rank_cell = agg_class(rank_arr)
    frac_cell = agg_class(frac_arr)
    z_cell    = agg_class(z_arr)
    p_cell    = agg_class(p_comb)
    fdr_cell  = agg_class(fdr_comb)
    z_abs_cell= agg_class(z_abs_arr)
    p_abs_cell= agg_class(p_comb_abs)
    fdr_abs_cell=agg_class(fdr_comb_abs)

    # Save
    adata.obsm["shapset_mean"]   = pd.DataFrame(mean_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_rankucell"]=pd.DataFrame(rank_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_frac"]  = pd.DataFrame(frac_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_z"]     = pd.DataFrame(z_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_pval"]  = pd.DataFrame(p_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_fdr"]   = pd.DataFrame(fdr_cell,index=adata.obs_names,columns=set_names_master)
    # NEW absolute variant
    adata.obsm["shapset_z_abs"]    = pd.DataFrame(z_abs_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_pval_abs"] = pd.DataFrame(p_abs_cell,index=adata.obs_names,columns=set_names_master)
    adata.obsm["shapset_fdr_abs"]  = pd.DataFrame(fdr_abs_cell,index=adata.obs_names,columns=set_names_master)

    return {"classes":class_names,"set_names":set_names_master,
            "scores_mean_cell":mean_cell,"scores_rank_cell":rank_cell,
            "scores_frac_cell":frac_cell,"scores_z_cell":z_cell,
            "scores_z_abs_cell":z_abs_cell}


#########


In [ ]:
import glob

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/"

In [ ]:
FList=glob.glob(dir+"norm*")
FList.sort()
FList

In [ ]:
FList=['/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_11.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_13.parquet',
  '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_14.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_15.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_17.parquet',

 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_18.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_19.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_20.parquet',
  '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_4.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_5.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_7.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_8.parquet']

In [ ]:
Numbs=[11,13,14,15,17,18,19,20,4,5,7,8]

In [ ]:
Numbs.sort()
Numbs

In [ ]:
DBs=[f"BCK{N}" for N in Numbs]
DBs

In [ ]:
for DB,N in zip(DBs,Numbs):
    print(DB)
    globals()[DB]=pd.read_parquet(f"/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_{N}.parquet")

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep

In [ ]:
for DB in DBs:
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
    try:
        globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)        
    except:
        pass




In [ ]:
N=list(globals()[DBs[0]].columns)
N.sort()
N.remove('N-cadherin')
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
for DB in DBs:
    globals()[DB]=globals()[DB][N]
#    globals()[DB]['Samp']=DB

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
#    globals()[DB]=np.arcsinh(globals()[DB]/5)
    globals()[DB]['Samp']=DB

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB].sample(10000,replace=False)]).copy()
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=None,verbose=True)

X_2d=UM.fit_transform(CAll[CellIden])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
CAll['x']=X_2d[:,0]
CAll['y']=X_2d[:,1]

In [ ]:
CAll.reset_index(drop=True,inplace=True)

In [ ]:
%matplotlib widget

In [ ]:
gdf=ManualSelection(CAll)

In [ ]:
CAll=gdf.copy()

In [ ]:
CAll

In [ ]:
for DB in DBs:
    M=CAll.Samp==DB
    globals()[DB]=CAll[M].copy()
    globals()[DB]=globals()[DB][globals()[DB].region_id!=0]
    globals()[DB]=globals()[DB][NamesAll]
    

In [ ]:
for DB in DBs:

    globals()[DB]['Samp']=DB
    print(DB,globals()[DB].shape[0])

In [ ]:
DF=pd.DataFrame([],columns=globals()[DBs[0]].columns)
NC=1000
for DB in DBs:
    print(DB)
    DF=pd.concat([DF,globals()[DB].sample(NC,replace=False)])
DF.reset_index(drop=True,inplace=True)

In [ ]:
DF=BCK18.copy()

In [ ]:
# ==== 1) Define gene/marker sets that match your panel names exactly ====
marker_sets = {
    # Epithelial & lineage
    "Epithelial_Luminal": {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"},
    "Basal_like": {"KRT5", "CD44", "Vimentin"},  # basal/intermediate cytokeratin + mesenchymal marker
    "Basal_Noa":{'H3K4me1', 'H3K9me2', 'H3K4me3'},
    # Stemness / tumor-initiating (mix of surface & chromatin)
#    "Stem_Prog": {"CD44", "CD24", "CD49f", "BMI1", "EZH2"},

    # EMT / mesenchymal programs (use signed weights below to penalize E-cadherin)
    "EMT": {"Vimentin", "aSMA", "CD44", "E-cadherin"},

    # Proliferation / cell cycle
    "Proliferation": {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},

    # DNA damage / repair
#    "DNA_Damage": {"pH2A.X", "H2AK119ub"},

    # Polycomb repression (PRC1/2)
#    "Polycomb_Repression": {"H3K27me3", "H3K27me2", "EZH2", "H2AK119ub"},

    # Enhancer activation
#    "Active_Enhancer": {"H3K27ac", "H3K4me1", "H3K64ac", "H4K16ac", "H3K9ac"},

    # Transcriptional elongation / gene body
#    "Elongation_H3K36": {"H3K36me3", "H3K36me2"},

    # Promoter activation/repression
#    "Promoter_Active": {"H3K4me3", "H3K9ac", "H3K27ac"},
#    "Promoter_Poised_Bivalent": {"H3K4me3", "H3K27me3"},  # bivalency

    # Heterochromatin / compaction
#    "Heterochromatin": {"H3K9me3", "H3K9me2", "H4K20me3"},

    # Broad histone cores (optional coarse sets)
#    "Core_Histones": {"H3", "H4", "H3.3"},
}

# ==== 2) Signed weights for mixed-direction sets (optional) ====
# These let you encode patterns like CD44^hi / CD24^lo or EMT up with E-cadherin down.
signed_weights = {
    # CD44 high, CD24 low
    "Stem_Prog": {"CD44": +1.0, "CD24": -1.0},
    # EMT up, E-cad down
    "EMT": {"Vimentin": +1.0, "aSMA": +1.0, "CD44": +0.5, "E-cadherin": -1.0},
}

In [ ]:
import umap

In [ ]:
MRK_All=N.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')


In [ ]:
UM=umap.UMAP(min_dist=0.01,n_neighbors=15,verbose=True)
X=UM.fit_transform(DF[MRK_All])

In [ ]:
DF

In [ ]:
features = MRK_All
metadata = ["Samp"]

adata = ad.AnnData(DF[features].values, obs=DF[metadata].copy())
adata.var_names = features
#adata.obs['Line']=adata.obs['Line'].astype("category")
adata.obsm["X_umap"] = X
for col in adata.obs.columns:
    adata.obs[col]=adata.obs[col].astype("category")

In [ ]:
%matplotlib inline

In [ ]:
sc.pl.umap(adata,color=MRK_All+['Samp'],cmap='seismic',vmin='p1',vmax='p99',show=False);
#plt.savefig("BRCA_UMAP.png",dpi=200,bbox_inches='tight')

In [ ]:
# ==== 3) (Optional) Quick overlap sanity check ====
panel = set(adata.var_names)
coverage = {k: sorted(list(panel.intersection(v))) for k, v in marker_sets.items()}
print("Per-set overlap with your panel (channels actually used):")
for k, v in coverage.items():
    print(f"  - {k}: {len(v)} features -> {v}")

In [ ]:
adata.layers['arcsinh']=adata.X

In [ ]:
# ============================================================
# SHAP-set (XGBoost) — FAST refactor w/ PyTorch (+MPS) & notebook bars
# ============================================================
# pip install numpy pandas anndata scanpy xgboost scikit-learn scipy torch tqdm

from __future__ import annotations
import time
from typing import Dict, Set, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.calibration import calibration_curve
from scipy.stats import chi2
from xgboost import XGBClassifier
import xgboost as xgb
from tqdm.auto import tqdm

# ---------- optional torch (for GPU/MPS accel) ----------
try:
    import torch
    _TORCH_OK = True
except Exception:
    _TORCH_OK = False


# =======================
# Device / dtype helpers
# =======================
def _select_device(prefer_mps: bool = True) -> str:
    if not _TORCH_OK:
        return "cpu"
    if prefer_mps and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

_TORCH_DEVICE = _select_device(prefer_mps=True)
_TORCH_DTYPE = torch.float32   # IMPORTANT: MPS prefers float32


# =======================
# Utility helpers (API kept)
# =======================
def get_layer_matrix(adata, layer: str):
    """Return matrix from AnnData layer or X (float32)."""
    X = adata.layers[layer] if (layer and layer in adata.layers) else adata.X
    return np.asarray(X, dtype=np.float32)

def intersect_sets(var_names, marker_sets: Dict[str, Set[str]]):
    """Intersect sets with current feature space; return names, mask [S,G], sizes [S]."""
    var_names = np.asarray(var_names)
    names, masks, sizes = [], [], []
    for nm, genes in marker_sets.items():
        m = np.isin(var_names, list(genes))
        if m.any():
            names.append(nm)
            masks.append(m)
            sizes.append(int(m.sum()))
    if not names:
        raise ValueError("No marker/gene sets overlap the features (var_names).")
    Gmask = np.vstack(masks).astype(bool)
    return names, Gmask, np.asarray(sizes, int)

def bh_fdr(p):
    """Benjamini–Hochberg FDR (vectorized)."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order]
    n = len(p)
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    return out


# =======================
# XGBoost contribs helpers (TreeSHAP)
# =======================
def _predict_contribs_compat(clf: XGBClassifier, X_block: np.ndarray,
                             feature_names=None, approx: bool = False) -> np.ndarray:
    """
    Robustly fetch SHAP (TreeSHAP) contributions from XGBoost, with bias column.
    Works across older/newer xgboost versions.
    """
    # Try sklearn wrapper (newer xgboost supports pred_contribs on sklearn API)
    try:
        return clf.predict(X_block, pred_contribs=True)  # may raise TypeError
    except TypeError:
        pass
    # Fallback to Booster API (most robust)
    dm = xgb.DMatrix(X_block, feature_names=(list(feature_names) if feature_names is not None else None))
    return clf.get_booster().predict(
        dm,
        pred_contribs=True,          # exact TreeSHAP on margins
        approx_contribs=approx,
        validate_features=True
    )

def _xgb_contribs_to_tensor(pred_contribs: np.ndarray, expected_n_features: int | None = None) -> np.ndarray:
    """
    Convert XGBoost pred_contribs output to (N, C, G) without bias.
    If expected_n_features is given, trim/pad to match.
    """
    arr = np.asarray(pred_contribs, dtype=np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Unexpected pred_contribs shape: {arr.shape}")
    A, B, C = arr.shape
    # Case A: (N, G+1, C)  -> drop bias, transpose -> (N, C, G)
    if C <= 64 and B >= 2:
        feats = B - 1
        out = np.transpose(arr[:, :feats, :], (0, 2, 1))  # (N, C, G)
    # Case B: (N, C, G+1)  -> drop bias -> (N, C, G)
    elif B <= 64 and C >= 2:
        feats = C - 1
        out = arr[:, :, :feats]
    else:
        raise ValueError(f"Can't interpret pred_contribs shape: {arr.shape}")

    if expected_n_features is not None and out.shape[2] != expected_n_features:
        Gc = out.shape[2]
        Ge = int(expected_n_features)
        if Gc > Ge:
            out = out[:, :, :Ge]  # trim
        elif Gc < Ge:
            pad = np.zeros((out.shape[0], out.shape[1], Ge - Gc), dtype=out.dtype)
            out = np.concatenate([out, pad], axis=2)
    return out


# =======================
# Torch-accelerated set scoring (MPS/CUDA/CPU)
# =======================
def _scores_for_sets_torch(
    SH_np: np.ndarray,     # (N, C, G), float32
    Gmask_np: np.ndarray,  # (S, G), bool
    sizes_np: np.ndarray,  # (S,)
    n_perm: int,
    signed_weights: np.ndarray | None = None,  # (S, G) float32 or None
    abs_variant: bool = True,
    seed: int | None = None,
    device: str = _TORCH_DEVICE,
) -> Tuple[np.ndarray, ...]:
    """
    Returns: set_mean, set_frac, set_rank, set_z, set_p, set_z_abs, set_p_abs
    Each shape (N, C, S). Abs arrays may be None if abs_variant=False.
    """
    if not _TORCH_OK:
        raise RuntimeError("PyTorch is required for the accelerated path.")

    torch.manual_seed(seed or 0)
    N, C, G = SH_np.shape
    S = Gmask_np.shape[0]

    SH = torch.from_numpy(SH_np).to(device=device, dtype=_TORCH_DTYPE)          # [N,C,G]
    Gmask = torch.from_numpy(Gmask_np).to(device=device)                         # [S,G] bool
    sizes = torch.from_numpy(sizes_np.astype(np.int32)).to(device=device)        # [S]
    SW = None
    if signed_weights is not None:
        SW = torch.from_numpy(signed_weights.astype(np.float32)).to(device=device)

    # (A) set sum / mean
    if SW is not None:
        set_sum_default = torch.einsum("ncg,sg->ncs", SH, Gmask.float())
        set_sum_signed = torch.einsum("ncg,sg->ncs", SH, SW)
        use_signed = (SW.abs().sum(dim=1) > 0).view(1, 1, S)  # [1,1,S]
        set_sum = torch.where(use_signed, set_sum_signed, set_sum_default)
    else:
        set_sum = torch.einsum("ncg,sg->ncs", SH, Gmask.float())

    set_mean = set_sum / torch.clamp(sizes.view(1, 1, S).float(), min=1.0)

    # (B) signed fraction
    denom = SH.abs().sum(dim=2, keepdim=True) + 1e-12
    set_frac = (set_sum / denom).to(_TORCH_DTYPE)

    # (C) rank-U
    # order = torch.argsort(-SH, dim=2)                                            # [N,C,G]
    # ranks = torch.empty_like(order)
    # arange_g = torch.arange(G, device=device).view(1, 1, -1)
    # ranks.scatter_(2, order, arange_g)
    # ranks = ranks + 1

    # (C) rank-U (inverse permutation via argsort-of-argsort; avoids scatter_ broadcasting issues)
    order = torch.argsort(-SH, dim=2)            # [N,C,G], each row is permutation of 0..G-1
    ranks = torch.argsort(order, dim=2) + 1      # [N,C,G], ranks in 1..G

    
    set_rank = torch.empty((N, C, S), device=device, dtype=_TORCH_DTYPE)
    m_const = (sizes * (sizes + 1) / 2.0).to(_TORCH_DTYPE)
    for s in range(S):
        m = int(sizes[s].item())
        if m == 0:
            set_rank[:, :, s] = 0
            continue
        idx = torch.nonzero(Gmask[s], as_tuple=False).view(-1)
        r = torch.index_select(ranks, 2, idx)
        U = r.sum(dim=2) - m_const[s]
        if m == G:
            set_rank[:, :, s] = 1.0
        else:
            set_rank[:, :, s] = (U / (m * (G - m))).to(_TORCH_DTYPE)

    # (D) permutation nulls (shared per unique m)
    uniq_m = torch.unique(sizes).tolist()
    set_z = torch.empty((N, C, S), device=device, dtype=_TORCH_DTYPE)
    set_p = torch.empty((N, C, S), device=device, dtype=torch.float32)
    set_z_abs = set_p_abs = None
    if abs_variant:
        set_z_abs = torch.empty_like(set_z)
        set_p_abs = torch.empty_like(set_p)

    SH_flat = SH.reshape(N * C, G)
    abs_SH_flat = SH_flat.abs() if abs_variant else None

    for m in uniq_m:
        m = int(m)
        if m <= 0:
            continue
        idxs = torch.stack([torch.randperm(G, device=device)[:m] for _ in range(n_perm)], dim=0)  # [P,m]

        base = SH_flat.unsqueeze(1).expand(-1, n_perm, -1)
        gathered = torch.gather(base, 2, idxs.unsqueeze(0).expand(N*C, -1, -1))
        null = gathered.sum(dim=2)                                               # [NC,P]
        mu = null.mean(dim=1, keepdim=True)
        sd = null.std(dim=1, unbiased=False, keepdim=True) + 1e-6

        if abs_variant:
            abase = abs_SH_flat.unsqueeze(1).expand(-1, n_perm, -1)
            agather = torch.gather(abase, 2, idxs.unsqueeze(0).expand(N*C, -1, -1))
            anull = agather.sum(dim=2)
            amu = anull.mean(dim=1, keepdim=True)
            asd = anull.std(dim=1, unbiased=False, keepdim=True) + 1e-6

        sets_of_m = torch.nonzero(sizes == m, as_tuple=False).view(-1)
        if sets_of_m.numel() == 0:
            continue

        for s in sets_of_m.tolist():
            obs = set_sum[:, :, s].reshape(N * C, 1)
            z = ((obs - mu) / sd).reshape(N, C)
            ge = (null >= obs).sum(dim=1).reshape(N, C)
            p = (ge + 1.0) / (n_perm + 1.0)
            set_z[:, :, s] = z.to(_TORCH_DTYPE)
            set_p[:, :, s] = p.to(torch.float32)

            if abs_variant:
                aobs = (set_sum[:, :, s].abs()).reshape(N * C, 1)
                az = ((aobs - amu) / asd).reshape(N, C)
                age = (anull >= aobs).sum(dim=1).reshape(N, C)
                ap = (age + 1.0) / (n_perm + 1.0)
                set_z_abs[:, :, s] = az.to(_TORCH_DTYPE)
                set_p_abs[:, :, s] = ap.to(torch.float32)

    def _to_np(x):
        return x.detach().to("cpu").numpy()

    out = (
        _to_np(set_mean),
        _to_np(set_frac),
        _to_np(set_rank),
        _to_np(set_z),
        _to_np(set_p),
        _to_np(set_z_abs) if abs_variant else None,
        _to_np(set_p_abs) if abs_variant else None,
    )
    # free memory
    del SH, Gmask, sizes, order, ranks
    return out


# =======================
# OOF metrics (same logic)
# =======================
def oof_classifier_metrics_xgb(X, y_int, n_splits=5, seed=0, model_kwargs=None):
    from sklearn.metrics import log_loss, confusion_matrix
    model_kwargs = model_kwargs or {}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    base = XGBClassifier(
        objective="multi:softprob",
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=0,
        **model_kwargs,
    )
    C = len(np.unique(y_int))
    probs = np.zeros((len(y_int), C), dtype=np.float32)
    preds = np.empty(len(y_int), dtype=int)
    for tr, te in skf.split(X, y_int):
        clf = XGBClassifier(**base.get_params())
        clf.set_params(random_state=int(np.random.default_rng(seed).integers(1, 2**31 - 1)))
        clf.fit(X[tr], y_int[tr])
        P = clf.predict_proba(X[te])
        probs[te] = P
        preds[te] = np.argmax(P, axis=1)
    macro_f1 = f1_score(y_int, preds, average="macro")
    micro_f1 = f1_score(y_int, preds, average="micro")
    Y = pd.get_dummies(y_int)
    try:
        macro_auroc = roc_auc_score(Y, probs, average="macro", multi_class="ovr")
    except ValueError:
        macro_auroc = np.nan
    ll = log_loss(y_int, probs)
    brier = float(np.mean([((y_int == i).astype(int) - probs[:, i]) ** 2 for i in range(C)]))
    conf = probs.max(1); acc = (preds == y_int).astype(int)
    fracs, means = calibration_curve(acc, conf, n_bins=10, strategy="quantile")
    ece = float(np.mean(np.abs(fracs - means)))
    cm = pd.DataFrame(
        confusion_matrix(y_int, preds, labels=np.arange(C)),
        index=[f"true_{i}" for i in range(C)],
        columns=[f"pred_{i}" for i in range(C)],
    )
    return {
        "classes_int": np.arange(C),
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_auroc": macro_auroc,
        "log_loss": ll,
        "brier": brier,
        "ECE": ece,
        "confusion_matrix": cm,
        "oof_probs": probs,
        "oof_preds": preds,
    }


# =======================
# Main FAST function (with Jupyter bars + robust alignment)
# =======================
def shap_cluster_gene_set_scores_xgb_fast(
    adata,
    marker_sets: Dict[str, Set[str]],
    layer: str = "arcsinh",
    cluster_key: str = "cluster",
    do_cluster: bool = False,
    leiden_res: float = 1.0,
    neighbors_k: int = 15,
    n_repeats: int = 2,
    n_splits: int = 5,
    n_perm: int = 200,
    random_state: int = 0,
    model_kwargs: dict | None = None,
    signed_weights: Dict[str, Dict[str, float]] | None = None,
    return_metrics: bool = True,
    class_agg: str = "prob",            # 'prob' | 'mean' | 'max' | 'pred'
    abs_variant: bool = True,           # compute abs(Z) permutation too
):
    """
    Fast SHAP-set with:
      - XGB TreeSHAP via pred_contribs (Booster fallback for compatibility)
      - Torch-accelerated set scoring (MPS/CUDA/CPU)
      - Defensive feature-axis alignment (avoids einsum size mismatch)
      - Jupyter-friendly progress bars
    """
    import scanpy as sc

    t0 = time.perf_counter()
    rng_master = np.random.default_rng(random_state)

    # Optional pseudo-label clustering
    if (cluster_key not in adata.obs) and do_cluster:
        rep = "X_cytovi" if "X_cytovi" in adata.obsm else None
        print(f"[info] Building neighbors (k={neighbors_k}) and Leiden (res={leiden_res})…")
        sc.pp.neighbors(adata, use_rep=rep, n_neighbors=neighbors_k)
        sc.tl.leiden(adata, key_added=cluster_key, resolution=leiden_res)

    # Encode labels
    y_str = adata.obs[cluster_key].astype(str).to_numpy()
    le = LabelEncoder().fit(y_str)
    y_int = le.transform(y_str)
    class_names = le.classes_
    C_master = len(class_names)

    # Feature matrix (float32)
    X_all = get_layer_matrix(adata, layer)
    genes_all = np.asarray(adata.var_names)

    # Restrict to union of set members (speed win)
    union = np.zeros(len(genes_all), bool)
    for S in marker_sets.values():
        union |= np.isin(genes_all, list(S))
    if union.sum() >= 5:
        X = X_all[:, union].astype(np.float32, copy=False)
        genes = genes_all[union]
    else:
        X = X_all.astype(np.float32, copy=False)
        genes = genes_all

    # Master sets (over full universe for stable column naming)
    set_names_master, _, _ = intersect_sets(genes_all, marker_sets)
    if len(set_names_master) == 0:
        raise ValueError("None of the marker sets overlap features.")
    S_master = len(set_names_master)
    name_to_master = {nm: i for i, nm in enumerate(set_names_master)}

    # CV base model
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    base = XGBClassifier(
        objective="multi:softprob",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=0,
        **(model_kwargs or {}),
    )

    # Optional signed weights over full universe
    signed_W_master = None
    if signed_weights is not None:
        signed_W_master = np.zeros((S_master, len(genes_all)), dtype=np.float32)
        for si, sname in enumerate(set_names_master):
            if sname in signed_weights:
                for feat, w in signed_weights[sname].items():
                    j = np.where(genes_all == feat)[0]
                    if j.size:
                        signed_W_master[si, j[0]] = np.float32(w)

    N, G = X.shape[0], X.shape[1]
    print(f"[start] device={_TORCH_DEVICE}  dtype=float32  N={N}  G={G}  C={C_master}  "
          f"repeats={n_repeats}  folds={n_splits}  perms={n_perm}")

    # Containers across repeats
    mean_runs, rank_runs, frac_runs, z_runs, p_runs = [], [], [], [], []
    z_abs_runs, p_abs_runs = [], []
    prob_runs, pred_runs = [], []

    with tqdm(total=n_repeats, desc="Repeats", position=0, leave=False, dynamic_ncols=True) as pbar_rep:
        for rep_idx in range(n_repeats):
            rng = np.random.default_rng(rng_master.integers(1, 2**31 - 1))

            mean_oof = np.zeros((N, C_master, S_master), np.float32)
            rank_oof = np.zeros((N, C_master, S_master), np.float32)
            frac_oof = np.zeros((N, C_master, S_master), np.float32)
            z_oof    = np.zeros((N, C_master, S_master), np.float32)
            p_oof    = np.ones((N, C_master, S_master),  np.float32)
            if abs_variant:
                z_oof_abs = np.zeros((N, C_master, S_master), np.float32)
                p_oof_abs = np.ones((N, C_master, S_master),  np.float32)

            oof_probs = np.zeros((N, C_master), np.float32)
            oof_pred  = np.zeros(N, dtype=int)

            with tqdm(total=n_splits, desc=f"CV folds (rep {rep_idx+1}/{n_repeats})",
                      position=1, leave=False, dynamic_ncols=True) as pbar_fold:

                for tr, te in skf.split(X, y_int):
                    clf = XGBClassifier(**base.get_params())
                    clf.set_params(random_state=int(rng.integers(1, 2**31 - 1)))
                    clf.fit(X[tr], y_int[tr])

                    # OOF probs/preds aligned to master class order
                    P = clf.predict_proba(X[te])  # (te, C_fold)
                    classes_fold = getattr(clf, "classes_", np.arange(P.shape[1]))
                    for i, c in enumerate(classes_fold):
                        c = int(c)
                        if 0 <= c < C_master:
                            oof_probs[te, c] = P[:, i]
                    oof_pred[te] = np.argmax(oof_probs[te], axis=1)

                    # ---- FAST SHAP via pred_contribs (TreeSHAP; margins) ----
                    contribs = _predict_contribs_compat(clf, X[te], feature_names=genes, approx=False)
                    SH = _xgb_contribs_to_tensor(contribs, expected_n_features=genes.shape[0])  # (te, C_fold, G_fold)

                    # ---- Defensive alignment of feature axis ----
                    feats_axis = genes                    # intended order used in training
                    G_from_shap = SH.shape[2]

                    if feats_axis.shape[0] != G_from_shap:
                        # enforce same length as SHAP contributions
                        if feats_axis.shape[0] > G_from_shap:
                            feats_axis_eff = feats_axis[:G_from_shap]
                        else:
                            Gmin = min(G_from_shap, feats_axis.shape[0])
                            feats_axis_eff = feats_axis[:Gmin]
                            SH = SH[:, :, :Gmin]
                    else:
                        feats_axis_eff = feats_axis

                    # Build mask & sizes on the effective feature list
                    fold_set_names, Gmask_fold, sizes_fold = intersect_sets(feats_axis_eff, marker_sets)
                    if len(fold_set_names) == 0:
                        pbar_fold.update(1)
                        continue
                    fold_to_master = np.array([name_to_master[nm] for nm in fold_set_names], int)

                    # Project optional signed weights to this axis
                    SW_fold = None
                    if signed_W_master is not None:
                        # map feats_axis_eff -> genes_all positions
                        # fast path: try searchsorted (requires sorted genes_all)
                        try:
                            pos_in_all = np.searchsorted(genes_all, feats_axis_eff)
                            if not np.all(genes_all[pos_in_all] == feats_axis_eff):
                                raise ValueError
                        except Exception:
                            pos_map = {g: i for i, g in enumerate(genes_all)}
                            pos_in_all = np.array([pos_map[g] for g in feats_axis_eff], int)
                        SW_fold = signed_W_master[fold_to_master][:, pos_in_all]  # [S_fold, G_eff]

                    # ---- Torch-accelerated scoring (with permutations) ----
                    set_mean, set_frac, set_rank, set_z, set_p, set_z_abs, set_p_abs = _scores_for_sets_torch(
                        SH_np=SH,
                        Gmask_np=Gmask_fold,
                        sizes_np=sizes_fold,
                        n_perm=n_perm,
                        signed_weights=SW_fold,
                        abs_variant=abs_variant,
                        seed=int(rng.integers(1, 2**31 - 1)),
                        device=_TORCH_DEVICE,
                    )

                    # Map to master class slots
                    C_fold = SH.shape[1]
                    if len(classes_fold) != C_fold:
                        classes_fold = np.arange(C_fold)
                    class_pos_sh = np.full(C_fold, -1, dtype=int)
                    for i, c in enumerate(classes_fold):
                        c = int(c)
                        if 0 <= c < C_master:
                            class_pos_sh[i] = c

                    for i_c, cpos in enumerate(class_pos_sh):
                        if cpos < 0:
                            continue
                        idx = np.ix_(te, [cpos], fold_to_master)
                        mean_oof[idx] = set_mean[:, i_c, :][:, None, :]
                        rank_oof[idx] = set_rank[:, i_c, :][:, None, :]
                        frac_oof[idx] = set_frac[:, i_c, :][:, None, :]
                        z_oof[idx]    = set_z[:, i_c, :][:, None, :]
                        p_oof[idx]    = set_p[:, i_c, :][:, None, :]
                        if abs_variant:
                            z_oof_abs[idx] = set_z_abs[:, i_c, :][:, None, :]
                            p_oof_abs[idx] = set_p_abs[:, i_c, :][:, None, :]

                    pbar_fold.update(1)

            mean_runs.append(mean_oof); rank_runs.append(rank_oof); frac_runs.append(frac_oof)
            z_runs.append(z_oof); p_runs.append(p_oof)
            if abs_variant:
                z_abs_runs.append(z_oof_abs); p_abs_runs.append(p_oof_abs)
            prob_runs.append(oof_probs); pred_runs.append(oof_pred)

            pbar_rep.update(1)

    # ===== Combine repeats =====
    mean_arr = np.stack(mean_runs, 0).mean(0)                 # [N,C,S]
    rank_arr = np.stack(rank_runs, 0).mean(0)
    frac_arr = np.stack(frac_runs, 0).mean(0)
    z_arr    = np.stack(z_runs, 0).mean(0)
    p_arr    = np.stack(p_runs, 0)                            # [R,N,C,S]

    stat = -2.0 * np.sum(np.log(np.clip(p_arr, 1e-300, 1.0)), axis=0)
    p_comb = 1.0 - chi2.cdf(stat, 2 * p_arr.shape[0])        # [N,C,S]
    fdr_comb = bh_fdr(p_comb.ravel()).reshape(p_comb.shape)

    if abs_variant:
        z_abs_arr = np.stack(z_abs_runs, 0).mean(0)
        p_abs_arr = np.stack(p_abs_runs, 0)
        stat_abs = -2.0 * np.sum(np.log(np.clip(p_abs_arr, 1e-300, 1.0)), axis=0)
        p_comb_abs = 1.0 - chi2.cdf(stat_abs, 2 * p_abs_arr.shape[0])
        fdr_comb_abs = bh_fdr(p_comb_abs.ravel()).reshape(p_comb_abs.shape)

    probs_mean = np.stack(prob_runs, 0).mean(0)               # [N,C]
    pred_mode  = np.round(np.stack(pred_runs, 0).mean(0)).astype(int)

    def agg_class(T: np.ndarray) -> np.ndarray:
        if T is None:
            return None
        if class_agg == "prob":
            Wc = probs_mean / (probs_mean.sum(1, keepdims=True) + 1e-12)
            return (T * Wc[:, :, None]).sum(1)
        elif class_agg == "mean":
            return T.mean(1)
        elif class_agg == "max":
            return T.max(1)
        elif class_agg == "pred":
            rows = np.arange(T.shape[0])
            return T[rows, pred_mode, :]
        else:
            raise ValueError("class_agg must be one of {'prob','mean','max','pred'}")

    mean_cell = agg_class(mean_arr)
    rank_cell = agg_class(rank_arr)
    frac_cell = agg_class(frac_arr)
    z_cell    = agg_class(z_arr)
    p_cell    = agg_class(p_comb)
    fdr_cell  = agg_class(fdr_comb)

    # Save to AnnData (cells × sets)
    adata.obsm["shapset_mean"]        = pd.DataFrame(mean_cell, index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_rankucell"]   = pd.DataFrame(rank_cell, index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_frac"]        = pd.DataFrame(frac_cell, index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_z"]           = pd.DataFrame(z_cell,    index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_pval"]        = pd.DataFrame(p_cell,    index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_fdr"]         = pd.DataFrame(fdr_cell,  index=adata.obs_names, columns=set_names_master)

    # Per-class matrices (kept)
    for j, cls_name in enumerate(class_names):
        adata.obsm[f"shapset_mean_{cls_name}"]      = pd.DataFrame(mean_arr[:, j, :], index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_rankucell_{cls_name}"] = pd.DataFrame(rank_arr[:, j, :], index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_frac_{cls_name}"]      = pd.DataFrame(frac_arr[:, j, :], index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_z_{cls_name}"]         = pd.DataFrame(z_arr[:, j, :],   index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_pval_{cls_name}"]      = pd.DataFrame(p_comb[:, j, :],  index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_fdr_{cls_name}"]       = pd.DataFrame(fdr_comb[:, j, :],index=adata.obs_names, columns=set_names_master)

    if abs_variant:
        z_abs_cell   = agg_class(z_abs_arr)
        p_abs_cell   = agg_class(p_comb_abs)
        fdr_abs_cell = agg_class(fdr_comb_abs)
        adata.obsm["shapset_z_abs"]    = pd.DataFrame(z_abs_cell,   index=adata.obs_names, columns=set_names_master)
        adata.obsm["shapset_pval_abs"] = pd.DataFrame(p_abs_cell,   index=adata.obs_names, columns=set_names_master)
        adata.obsm["shapset_fdr_abs"]  = pd.DataFrame(fdr_abs_cell, index=adata.obs_names, columns=set_names_master)

    clf_metrics = None
    if return_metrics:
        clf_metrics = oof_classifier_metrics_xgb(
            X, y_int, n_splits=n_splits, seed=random_state, model_kwargs=model_kwargs
        )

    dt = time.perf_counter() - t0
    print(f"[done] SHAP-set complete in {dt:.1f}s (device={_TORCH_DEVICE})")

    out = {
        "classes": class_names,
        "set_names": set_names_master,
        "scores_mean": mean_arr, "scores_rank": rank_arr,
        "scores_frac": frac_arr, "scores_z": z_arr,
        "pvals_cell_classwise": p_comb, "fdr_cell_classwise": fdr_comb,
        "scores_mean_cell": mean_cell, "scores_rank_cell": rank_cell,
        "scores_frac_cell": frac_cell, "scores_z_cell": z_cell,
        "pvals_cell": p_cell, "fdr_cell": fdr_cell,
        "class_agg": class_agg,
        "device": _TORCH_DEVICE,
        "elapsed_sec": dt,
    }
    if abs_variant:
        out.update({
            "scores_z_abs_cell": adata.obsm["shapset_z_abs"].to_numpy(),
            "pvals_abs_cell":    adata.obsm["shapset_pval_abs"].to_numpy(),
            "fdr_abs_cell":      adata.obsm["shapset_fdr_abs"].to_numpy(),
        })
    if clf_metrics is not None:
        out["clf_metrics"] = clf_metrics
    return out


In [ ]:
if "cluster" in adata.obs.columns:
    adata.obs.drop(columns=["cluster"], inplace=True)
out = shap_cluster_gene_set_scores_xgb_fast(
    adata,
    marker_sets,
    layer="arcsinh",           # or your preferred layer
    cluster_key="cluster",     # labels for the surrogate task (used to learn SHAP)
    n_splits=5,leiden_res=0.5,
    do_cluster=True,
    n_repeats=2,
    n_perm=200,
    random_state=1,
    signed_weights=None,
    class_agg="prob",          # <-- probability-weighted across classes -> cells × sets
    return_metrics=True,
)

In [ ]:
Z      = adata.obsm["shapset_z_abs"]          # permutation Z-scores,    cells × sets
Q      = adata.obsm["shapset_fdr"]        # FDR per cell × set
RANK   = adata.obsm["shapset_rankucell"]  # rank-based (0..1)
FRAC   = adata.obsm["shapset_frac"]       # signed fraction (−1..1)
MEAN   = adata.obsm["shapset_mean"]       # mean SHAP contribution
PV     = adata.obsm["shapset_pval"]       # raw permutation p-values

print("\nExample: top 5 cells by Active_Enhancer Z-score")
print(Z["Basal_Noa"].sort_values(ascending=False).head(5))


In [ ]:
import scanpy as sc
GS=list(marker_sets.keys())

In [ ]:
Calc=[
 'shapset_mean',
 'shapset_rankucell',
 'shapset_frac',
 'shapset_z',
 'shapset_z_abs',
]

In [ ]:
# Pick a gene set name, e.g. "Active_Enhancer"
for C in Calc:
    for gene_set in GS:
        # Put the scores into .obs so scanpy can color by them
        adata.obs[f"{gene_set}"] = adata.obsm[f"{C}"][gene_set]
        
        # Plot UMAP colored by Z-score
    sc.pl.umap(adata, color=GS+['Samp'], cmap="seismic",vmin='p1',vmax='p99',show=False,)
    plt.suptitle(f"{C}", fontsize=20, y=1,x=0)
    plt.show()
#    plt.savefig(f"BRCA_UMAP_MarkerSets_{C}.png",dpi=200,bbox_inches='tight')

In [ ]:
Z = adata.obsm["shapset_z"]       # cells × sets
P = adata.obsm["shapset_pval"]    # cells × sets

# sets with all-zero Z
zero_sets = [s for s in Z.columns if np.allclose(Z[s].values, 0)]

# if p-values are ~1 everywhere too, it's almost surely “no overlap” (case 1)
no_overlap_like = [s for s in zero_sets if np.allclose(P[s].values, 1)]

# check raw overlap with your panel & with the model’s feature matrix
panel = set(map(str, adata.var_names))
for s in zero_sets:
    overlap_panel = panel.intersection(set(marker_sets[s]))
    print(f"{s}: overlap with adata.var_names = {sorted(overlap_panel)}")


In [ ]:
import numpy as np, pandas as pd

Z = adata.obsm["shapset_z"]
P = adata.obsm["shapset_pval"]

for s in ["Basal_Noa", "Basal_like"]:
    print(s, "Z mean±sd:", float(Z[s].mean()), float(Z[s].std()), 
          "| any p<0.05?", bool((P[s] < 0.05).any()))


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

def shapset_resolution_ensemble(
    adata, marker_sets, resolutions=(0.4, 0.8, 1.2, 1.6),
    layer="arcsinh", neighbors_k=15, n_splits=5, n_repeats=2,
    n_perm=200, random_state=1, signed_weights=None, class_agg="prob"
):
    Zs, Qs, RANKs, FRACs, MEANs = [], [], [], [], []

    # keep a clean neighbors graph for reproducibility
    sc.pp.neighbors(adata, n_neighbors=neighbors_k, use_rep=None)

    for r in resolutions:
        # make labels at this resolution
        sc.tl.leiden(adata, key_added=f"leiden_r{r}", resolution=r)
        out = shap_cluster_gene_set_scores_xgb_fast(
            adata,
            marker_sets,
            layer=layer,
            cluster_key=f"leiden_r{r}",
            do_cluster=False,                # labels already created
            n_splits=n_splits, n_repeats=n_repeats,
            n_perm=n_perm, random_state=random_state,
            signed_weights=signed_weights,
            class_agg=class_agg,            # prob-weighted across classes
            return_metrics=False,
        )
        Zs.append(adata.obsm["shapset_z"].copy())
        Qs.append(adata.obsm["shapset_fdr"].copy())
        RANKs.append(adata.obsm["shapset_rankucell"].copy())
        FRACs.append(adata.obsm["shapset_frac"].copy())
        MEANs.append(adata.obsm["shapset_mean"].copy())

    # stack and aggregate (median is robust)
    Z_med   = pd.concat(Zs,   axis=1, keys=range(len(Zs))).groupby(level=1, axis=1).median()
    Q_med   = pd.concat(Qs,   axis=1, keys=range(len(Qs))).groupby(level=1, axis=1).median()
    R_med   = pd.concat(RANKs,axis=1, keys=range(len(RANKs))).groupby(level=1, axis=1).median()
    F_med   = pd.concat(FRACs,axis=1, keys=range(len(FRACs))).groupby(level=1, axis=1).median()
    M_med   = pd.concat(MEANs,axis=1, keys=range(len(MEANs))).groupby(level=1, axis=1).median()

    # save ensemble results
    adata.obsm["shapset_z_ens"]      = Z_med
    adata.obsm["shapset_fdr_ens"]    = Q_med
    adata.obsm["shapset_rank_ens"]   = R_med
    adata.obsm["shapset_frac_ens"]   = F_med
    adata.obsm["shapset_mean_ens"]   = M_med
    return {"Z_med": Z_med, "Q_med": Q_med}


In [ ]:
_ = shapset_resolution_ensemble(
    adata, marker_sets,
    resolutions=(0.4, 0.6,0.8, 1.0,1.2, 1.4,1.6),
    neighbors_k=15, class_agg="prob",
    signed_weights=None
)


In [ ]:
# Pick a gene set name, e.g. "Active_Enhancer"
for C in ["shapset_z_ens"]:
    for gene_set in GS:
        # Put the scores into .obs so scanpy can color by them
        adata.obs[f"{gene_set}"] = adata.obsm[f"{C}"][gene_set]
        
        # Plot UMAP colored by Z-score
    sc.pl.umap(adata, color=GS+['Samp'], cmap="seismic",vmin='p1',vmax='p99',show=False,)
    plt.suptitle(f"{C}", fontsize=20, y=.92)

    plt.savefig(f"BRCA_UMAP_MarkerSets_{C}.png",dpi=200,bbox_inches='tight')
    plt.show()    

In [ ]:
DF=adata.obsm["shapset_z_ens"].copy()
X2d=adata.obsm['X_umap'].copy()

In [ ]:
def gaussian_smooth_all(data, positions, bandwidth):
    """
    Perform Gaussian smoothing on all columns in the dataset based on 2D positions.

    Parameters:
    - data: np.ndarray, the dataset (rows = points, columns = features to be smoothed).
    - positions: np.ndarray, the 2D positions (rows correspond to the data rows, columns = [x, y]).
    - bandwidth: float, the standard deviation of the Gaussian kernel.

    Returns:
    - smoothed_data: np.ndarray, the smoothed dataset (same shape as `data`).
    """
    # Compute pairwise distances based on 2D positions
    distances = cdist(positions, positions)

    if bandwidth==-1:
        bandwidth=np.quantile(distances[np.triu_indices(distances.shape[0], k = 1)],0.05)

    print(bandwidth)
    # Compute Gaussian weights
    weights = norm.pdf(distances, scale=bandwidth)

    # Normalize weights so they sum to 1 for each point
    normalized_weights = weights / weights.sum(axis=1, keepdims=True)

    # Smooth each column
    smoothed_data = np.dot(normalized_weights, data)

    return smoothed_data

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.stats import norm


positions = X2d
# Data to be smoothed (rows = points, columns = features)
data = DF.values.copy()

# Perform Gaussian smoothing
bandwidth = -1
smoothed_data = gaussian_smooth_all(data, positions, bandwidth)
smoothed_df=pd.DataFrame(smoothed_data,columns=DF.columns)

In [ ]:
for C in DF.columns:
    plt.figure(figsize=(6,5))
    cc=smoothed_df[C].values
    M=cc>-1
    vmx=np.quantile(cc,0.99)
    vmn=np.quantile(cc,0.01)
#    plt.scatter(X_2d[M,0],X_2d[M,1],s=1, c='gray')
    plt.scatter(X2d[M,0],X2d[M,1],s=1, c=cc[M],cmap=plt.cm.magma_r,vmax=vmx,vmin=vmn)
    plt.colorbar()
    plt.title(C)

In [ ]:
# ==== 1) Define gene/marker sets that match your panel names exactly ====
marker_sets = {
    # Epithelial & lineage
    "Epithelial_Luminal": {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"},
    "Basal_like": {"KRT5", "CD44", "Vimentin"},  # basal/intermediate cytokeratin + mesenchymal marker
    "Basal_Noa":{'H3K4me1', 'H3K9me2', 'H3K4me3'},
    # Stemness / tumor-initiating (mix of surface & chromatin)
#    "Stem_Prog": {"CD44", "CD24", "CD49f", "BMI1", "EZH2"},

    # EMT / mesenchymal programs (use signed weights below to penalize E-cadherin)
    "EMT": {"Vimentin", "aSMA", "CD44", "E-cadherin"},
    # Proliferation / cell cycle
    "Proliferation": {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},
    # DNA damage / repair
    "DNA_Damage": {"pH2A.X", "H2AK119ub"},

    # # Polycomb repression (PRC1/2)
    # "Polycomb_Repression": {"H3K27me3", "H3K27me2", "EZH2", "H2AK119ub"},

    # # Enhancer activation
    # "Active_Enhancer": {"H3K27ac", "H3K4me1", "H3K64ac", "H4K16ac", "H3K9ac"},

    # # Transcriptional elongation / gene body
    # "Elongation_H3K36": {"H3K36me3", "H3K36me2"},

    # # Promoter activation/repression
    # "Promoter_Active": {"H3K4me3", "H3K9ac", "H3K27ac"},
    # "Promoter_Poised_Bivalent": {"H3K4me3", "H3K27me3"},  # bivalency

    # # Heterochromatin / compaction
    # "Heterochromatin": {"H3K9me3", "H3K9me2", "H4K20me3"},

    # Broad histone cores (optional coarse sets)
#    "Core_Histones": {"H3", "H4", "H3.3"},
}



In [ ]:
# ================== JUPYTER-SAFE, FLOAT32-SAFE VERSIONS ==================
import numpy as np
import pandas as pd

# Use notebook tqdm to avoid line spam; nested bars use positions.
try:
    from tqdm.notebook import tqdm
except Exception:
    # Fallback if notebook tqdm unavailable
    from tqdm import tqdm

# ---------------------------
# Torch helpers (MPS/CUDA/CPU)
# ---------------------------
def _torch_device():
    import torch
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _to_torch(x, device):
    import torch
    if isinstance(x, torch.Tensor):
        return x.to(device, dtype=torch.float32)
    return torch.as_tensor(x, device=device, dtype=torch.float32)

def _standardize_torch(X):
    import torch
    X = X.to(dtype=torch.float32)
    mu = X.mean(0, keepdim=True)
    sd = X.std(0, keepdim=True).clamp_min(1e-6)
    return (X - mu) / sd

@torch.no_grad()
def _knn_topk_torch(X, k, device=None, block=4096, desc=None, position=0, leave=False):
    """
    Chunked kNN via torch.cdist, returning indices [N, k] (excluding self).
    All ops in float32. Uses tqdm.notebook with position control.
    """
    import torch
    device = device or _torch_device()
    X = _to_torch(X, device).to(dtype=torch.float32)
    N = X.shape[0]
    k = int(min(k, max(1, N-1)))
    idx_all = torch.empty((N, k), dtype=torch.long, device=device)

    bar = tqdm(range(0, N, block), desc=desc or "kNN blocks", position=position, leave=leave)
    for i0 in bar:
        i1 = min(N, i0 + block)
        D = torch.cdist(X[i0:i1], X)  # float32
        rows = torch.arange(i0, i1, device=device)
        D[torch.arange(i1 - i0, device=device), rows] = float("inf")
        _, inds = torch.topk(D, k=k, largest=False, dim=1)
        idx_all[i0:i1] = inds
        del D, inds
        if device.type == "mps":
            torch.mps.empty_cache()
    return idx_all.detach().cpu().numpy()

# ---------------------------
# Utility
# ---------------------------
def _get_layer_block(adata, layer, genes):
    idx = adata.var_names.get_indexer([g for g in genes if g in adata.var_names])
    idx = idx[idx >= 0]
    if idx.size == 0:
        return None, []
    X = adata.layers[layer][:, idx] if (layer in adata.layers) else adata.X[:, idx]
    return np.asarray(X, dtype=np.float32), list(adata.var_names[idx])

def _rank_norm_cols(df):
    """Rank-normalize each column to [0,1] (ties averaged). Cast to float32 at end."""
    out = df.rank(method="average", pct=True)
    return out.astype(np.float32)

# ---------------------------
# FAST consensus per-cell (multiscale Leiden) — notebook-friendly
# ---------------------------
def per_cell_consensus_fast(
    adata,
    marker_sets: dict,
    layer: str = "arcsinh",
    k_grid=(10, 15, 30),
    gamma_grid=(0.4, 0.8, 1.2),
    n_repeats: int = 5,
    sample_frac: float = 0.9,
    k_eval: int = 20,
    random_state: int = 1,
):
    """
    Writes: adata.obsm["set_sep_consensus_fast"]  (cells × sets, float32)
    Jupyter-safe tqdm bars: one per set + inner per-set run counter; no line spam.
    """
    try:
        import igraph as ig, leidenalg as la
        have_leiden = True
    except Exception:
        have_leiden = False
        tqdm.write("[consensus] WARNING: 'leidenalg' not available; falling back to trivial labels.")

    rng = np.random.default_rng(random_state)
    X_all = adata.layers[layer] if layer in adata.layers else adata.X
    X_all = np.asarray(X_all, dtype=np.float32)
    N = X_all.shape[0]

    device = _torch_device()
    k_base = int(max(max(k_grid), k_eval))
    set_names, results = [], []

    tqdm.write(f"[consensus] sets={len(marker_sets)} | k_grid={k_grid} | gamma_grid={gamma_grid} | "
               f"repeats={n_repeats} | sample_frac={sample_frac} | k_eval={k_eval} | device={device.type}")

    for s_idx, (set_name, genes) in enumerate(tqdm(marker_sets.items(), desc="Sets (consensus)", position=0, leave=True)):
        X, used = _get_layer_block(adata, layer, genes)
        set_names.append(set_name)
        if X is None or X.shape[1] < 2 or N < 5:
            tqdm.write(f"[consensus] {set_name}: <2 features or N<5 → zeros")
            results.append(np.zeros(N, dtype=np.float32))
            continue

        # standardize + build base kNN once (float32)
        X_t = _to_torch(X, device)
        X_t = _standardize_torch(X_t)
        nn = _knn_topk_torch(X_t, k=k_base, device=device,
                             desc=f"kNN {set_name}", position=1, leave=False).astype(np.int32)
        del X_t

        nb_eval = nn[:, :min(k_eval, nn.shape[1])]  # [N, k_eval_eff]

        # Precompute undirected edges for each k
        edges_by_k = {}
        for k in k_grid:
            j_k = nn[:, :k].reshape(-1)
            i_k = np.repeat(np.arange(N, dtype=np.int32), k)
            Ek = np.vstack([np.stack([i_k, j_k], 1),
                            np.stack([j_k, i_k], 1)])
            Ek = Ek[Ek[:, 0] != Ek[:, 1]]
            flip = Ek[:, 0] > Ek[:, 1]
            Ek[flip] = Ek[flip][:, ::-1]
            Ek = np.unique(Ek, axis=0)
            edges_by_k[k] = Ek

        sums = np.zeros(N, dtype=np.float32)
        counts = np.zeros(N, dtype=np.int32)

        total_runs = n_repeats * len(gamma_grid) * len(k_grid)
        pbar = tqdm(total=total_runs, desc=f"Consensus [{set_name}]", position=2, leave=False)
        for rep in range(n_repeats):
            mask = rng.random(N) < sample_frac
            if mask.sum() < 10:
                pbar.update(len(gamma_grid) * len(k_grid))
                continue
            for gamma in gamma_grid:
                for k in k_grid:
                    if not have_leiden:
                        lab = np.full(N, -1, dtype=np.int32); lab[mask] = 0
                    else:
                        keep_idx = np.where(mask)[0]
                        local_id = -np.ones(N, dtype=np.int32)
                        local_id[keep_idx] = np.arange(keep_idx.size, dtype=np.int32)
                        Ek = edges_by_k[k]
                        sel = mask[Ek[:, 0]] & mask[Ek[:, 1]]
                        if not np.any(sel) or keep_idx.size < 5:
                            lab = np.full(N, -1, dtype=np.int32)
                        else:
                            e_sub = Ek[sel]
                            e_local = np.stack([local_id[e_sub[:, 0]], local_id[e_sub[:, 1]]], 1)
                            g = ig.Graph(n=keep_idx.size, edges=[tuple(x) for x in e_local], directed=False)
                            g.simplify(combine_edges="first")
                            part = la.find_partition(g, la.RBConfigurationVertexPartition,
                                                     resolution_parameter=float(gamma))
                            lab_sub = np.array(part.membership, dtype=np.int32)
                            lab = np.full(N, -1, dtype=np.int32); lab[keep_idx] = lab_sub

                    in_mask = np.where(mask)[0]
                    nb = nb_eval[in_mask]
                    valid = mask[nb]
                    lab_i = lab[in_mask][:, None]
                    lab_nb = lab[nb]
                    same = (lab_nb == lab_i) & valid
                    denom = valid.sum(1)
                    frac = np.divide(same.sum(1, dtype=np.int32), denom,
                                     out=np.zeros_like(denom, dtype=np.float32), where=denom > 0).astype(np.float32)
                    sums[in_mask] += frac
                    counts[in_mask] += 1
                    pbar.update(1)
        pbar.close()

        score = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0).astype(np.float32)
        results.append(np.nan_to_num(score, nan=0.0).astype(np.float32))

        # free MPS cache
        import torch
        if device.type == "mps":
            torch.mps.empty_cache()

    df = pd.DataFrame(np.vstack(results).T.astype(np.float32), index=adata.obs_names, columns=set_names)
    adata.obsm["set_sep_consensus_fast"] = df
    tqdm.write("[consensus] done ✓")
    return df

# ---------------------------
# FAST purity-Z per cell (clustering-free) — notebook-friendly
# ---------------------------
@torch.no_grad()
def per_cell_purityZ_fast(
    adata,
    marker_sets: dict,
    layer: str = "arcsinh",
    ks=(10, 20, 30),
    n_perm: int = 32,
    random_state: int = 1,
):
    """
    Writes: adata.obsm["set_sep_purityZ_fast"]  (cells × sets, float32)
    """
    import torch
    rng = np.random.default_rng(random_state)
    device = _torch_device()

    X_all = adata.layers[layer] if layer in adata.layers else adata.X
    X_all = np.asarray(X_all, dtype=np.float32)
    N = X_all.shape[0]

    set_names = []
    PZ_cols = []

    tqdm.write(f"[purityZ] sets={len(marker_sets)} | ks={ks} | perms={n_perm} | device={device.type}")

    for s_idx, (set_name, genes) in enumerate(tqdm(marker_sets.items(), desc="Sets (purityZ)", position=0, leave=True)):
        X, used = _get_layer_block(adata, layer, genes)
        set_names.append(set_name)
        if X is None or X.shape[1] < 2 or N < 5:
            tqdm.write(f"[purityZ] {set_name}: <2 features or N<5 → zeros")
            PZ_cols.append(np.zeros(N, dtype=np.float32))
            continue

        X_t = _to_torch(X, device)          # float32
        X_t = _standardize_torch(X_t)       # float32

        z_list = []
        for k in tqdm(ks, desc=f"{set_name}: ks", position=1, leave=False):
            nbr_obs = _knn_topk_torch(X_t, k=k, device=device, desc=f"kNN {set_name} (k={k})", position=2, leave=False)
            # inverse sets
            back_sets = [set() for _ in range(N)]
            for i in range(N):
                for j in nbr_obs[i]:
                    back_sets[j].add(int(i))
            pur_obs = np.fromiter(
                (np.mean([1.0 if i in back_sets[j] else 0.0 for j in nbr_obs[i]]) for i in range(N)),
                dtype=np.float32, count=N
            )

            # null perms
            null = np.empty((N, n_perm), dtype=np.float32)
            G = X_t.shape[1]
            pbar_perm = tqdm(range(n_perm), desc=f"{set_name}: perms(k={k})", position=3, leave=False)
            for p in pbar_perm:
                perm = torch.randperm(G, device=device)
                Xp = X_t[:, perm]  # float32
                nbr = _knn_topk_torch(Xp, k=k, device=device, desc=None, position=4, leave=False)
                back = [set() for _ in range(N)]
                for i in range(N):
                    for j in nbr[i]:
                        back[j].add(int(i))
                null[:, p] = np.fromiter(
                    (np.mean([1.0 if i in back[j] else 0.0 for j in nbr[i]]) for i in range(N)),
                    dtype=np.float32, count=N
                )
                if device.type == "mps":
                    torch.mps.empty_cache()

            mu = null.mean(1).astype(np.float32)
            sd = (null.std(1) + 1e-6).astype(np.float32)
            z_k = ((pur_obs - mu) / sd).astype(np.float32)
            z_list.append(z_k.astype(np.float32))

        z_mean = np.mean(np.vstack(z_list).astype(np.float32), axis=0, dtype=np.float32)
        PZ_cols.append(z_mean.astype(np.float32))

        del X_t
        if device.type == "mps":
            torch.mps.empty_cache()

    df = pd.DataFrame(np.vstack(PZ_cols).T.astype(np.float32), index=adata.obs_names, columns=set_names)
    adata.obsm["set_sep_purityZ_fast"] = df
    tqdm.write("[purityZ] done ✓")
    return df

# ---------------------------
# Optional HDBSCAN membership strength — notebook-friendly
# ---------------------------
def per_cell_hdbscan_strength(
    adata,
    marker_sets: dict,
    layer: str = "arcsinh",
    min_cluster_size: int = 30,
    min_samples=None,
    standardize: bool = True,
):
    """
    Writes: adata.obsm["set_sep_hdbscan_fast"] (cells × sets, float32)
    """
    try:
        import hdbscan
        have = True
    except Exception:
        have = False
        tqdm.write("[hdbscan] WARNING: package not installed; returning zeros.")

    set_names, HDB_cols = [], []
    X_all = adata.layers[layer] if layer in adata.layers else adata.X
    X_all = np.asarray(X_all, dtype=np.float32)

    tqdm.write(f"[hdbscan] sets={len(marker_sets)} | min_cluster_size={min_cluster_size} | min_samples={min_samples}")
    for set_name, genes in tqdm(marker_sets.items(), desc="Sets (hdbscan)", position=0, leave=True):
        set_names.append(set_name)
        idx = adata.var_names.get_indexer([g for g in genes if g in adata.var_names])
        idx = idx[idx >= 0]
        if idx.size < 2 or not have:
            HDB_cols.append(np.zeros(adata.n_obs, dtype=np.float32))
            continue
        X = X_all[:, idx]
        if standardize:
            from sklearn.preprocessing import StandardScaler
            X = StandardScaler(with_mean=True, with_std=True).fit_transform(X.astype(np.float32)).astype(np.float32)
        clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,
                                    min_samples=min_samples, prediction_data=True)
        clusterer.fit(X)
        HDB_cols.append(clusterer.probabilities_.astype(np.float32))

    df = pd.DataFrame(np.vstack(HDB_cols).T.astype(np.float32), index=adata.obs_names, columns=set_names)
    adata.obsm["set_sep_hdbscan_fast"] = df
    tqdm.write("[hdbscan] done ✓")
    return df

# ---------------------------
# Wrapper — notebook-friendly orchestration
# ---------------------------
def compute_marker_set_separability(
    adata,
    marker_sets: dict,
    layer: str = "arcsinh",
    # consensus params
    do_consensus: bool = True,
    k_grid=(10, 15, 30),
    gamma_grid=(0.4, 0.8, 1.2),
    n_repeats: int = 5,
    sample_frac: float = 0.9,
    k_eval: int = 20,
    # purity Z params
    do_purityZ: bool = True,
    ks=(10, 20, 30),
    n_perm: int = 32,
    # hdbscan
    do_hdbscan: bool = True,
    min_cluster_size: int = 30,
    min_samples=None,
    # general
    random_state: int = 1,
):
    outs = {}
    tqdm.write("[separability] starting... (Jupyter-safe, float32)")

    if do_consensus:
        outs["consensus"] = per_cell_consensus_fast(
            adata, marker_sets, layer=layer,
            k_grid=k_grid, gamma_grid=gamma_grid,
            n_repeats=n_repeats, sample_frac=sample_frac, k_eval=k_eval,
            random_state=random_state,
        ).astype(np.float32)

    if do_purityZ:
        outs["purityZ"] = per_cell_purityZ_fast(
            adata, marker_sets, layer=layer,
            ks=ks, n_perm=n_perm, random_state=random_state,
        ).astype(np.float32)

    if do_hdbscan:
        outs["hdbscan"] = per_cell_hdbscan_strength(
            adata, marker_sets, layer=layer,
            min_cluster_size=min_cluster_size, min_samples=min_samples,
        ).astype(np.float32)

    # Combine into final
    mats = []
    if "consensus" in outs: mats.append(_rank_norm_cols(outs["consensus"]))
    if "purityZ"   in outs: mats.append(_rank_norm_cols(outs["purityZ"]))
    if "hdbscan"   in outs: mats.append(_rank_norm_cols(outs["hdbscan"]))

    if mats:
        df_final = pd.concat([m.add_suffix(f"|comp{i}") for i, m in enumerate(mats)], axis=1) \
                    .groupby(lambda c: c.split("|")[0], axis=1).median()
    else:
        df_final = pd.DataFrame(index=adata.obs_names, columns=list(marker_sets.keys()), dtype=np.float32)
    df_final = df_final.astype(np.float32)
    adata.obsm["set_sep_final_fast"] = df_final
    outs["final"] = df_final

    tqdm.write("[separability] all done ✓")
    return outs

# ---------------------------
# Convenience plotting
# ---------------------------
def plot_set_score_umap(adata, set_name, obsm_key="set_sep_final_fast", vmin=None, vmax=None, cmap="magma"):
    import scanpy as sc
    if obsm_key not in adata.obsm:
        raise KeyError(f"{obsm_key} not found in adata.obsm")
    if set_name not in adata.obsm[obsm_key].columns:
        raise KeyError(f"{set_name} not found in adata.obsm['{obsm_key}'].columns")
    col = f"{obsm_key}:{set_name}"
    adata.obs[col] = adata.obsm[obsm_key][set_name].astype(np.float32)
    sc.pl.umap(adata, color=col, cmap=cmap, vmin=vmin, vmax=vmax)


In [ ]:
out=compute_marker_set_separability(adata,marker_sets,do_consensus=False, do_purityZ=False)

In [ ]:
adata

In [ ]:
plot_set_score_umap(adata,"Basal_like","set_sep_hdbscan_fast")